 01_results_analysis_vllm — normalize vLLM benchmark outputs, compute metrics, export processed tables

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW_RESULTS_DIR = ROOT / "results" / "raw"
PROCESSED_DIR = ROOT / "results" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["mmlu", "hellaswag", "gsm8k", "dolly"]

print("ROOT:", ROOT)
print("RAW_RESULTS_DIR:", RAW_RESULTS_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)


ROOT: /home/stephane/resource-efficiency-benchmark
RAW_RESULTS_DIR: /home/stephane/resource-efficiency-benchmark/results/raw
PROCESSED_DIR: /home/stephane/resource-efficiency-benchmark/results/processed


 Discover vLLM runs and load raw files

In [2]:
def discover_runs(results_dir: Path) -> pd.DataFrame:
    rows = []

    if not results_dir.exists():
        return pd.DataFrame(columns=["model_alias_dir", "run_id", "run_dir"])

    for model_dir in sorted(results_dir.iterdir()):
        if not model_dir.is_dir():
            continue

        for run_dir in sorted(model_dir.iterdir()):
            if not run_dir.is_dir():
                continue

            rows.append({
                "model_alias_dir": model_dir.name,
                "run_id": run_dir.name,
                "run_dir": str(run_dir),
            })

    return pd.DataFrame(rows)


def safe_read_csv(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path)
    except Exception as e:
        print(f"FAILED reading {path}: {e}")
        return pd.DataFrame()


def load_raw_outputs(results_dir: Path):
    runs_df = discover_runs(results_dir)

    summary_tables = []
    sample_tables = []

    for _, r in runs_df.iterrows():
        run_dir = Path(r["run_dir"])

        for ds in DATASETS:
            summary_path = run_dir / f"{ds}.csv"
            samples_path = run_dir / f"{ds}_samples.csv"

            if summary_path.exists():
                df = safe_read_csv(summary_path)
                if not df.empty:
                    df["model_alias_dir"] = r["model_alias_dir"]
                    df["run_id"] = r["run_id"]
                    df["dataset"] = ds
                    df["source_path"] = str(summary_path)
                    summary_tables.append(df)

            if samples_path.exists():
                df = safe_read_csv(samples_path)
                if not df.empty:
                    df["model_alias_dir"] = r["model_alias_dir"]
                    df["run_id"] = r["run_id"]
                    df["dataset"] = ds
                    df["source_path"] = str(samples_path)
                    sample_tables.append(df)

    raw_summary = pd.concat(summary_tables, ignore_index=True) if summary_tables else pd.DataFrame()
    raw_samples = pd.concat(sample_tables, ignore_index=True) if sample_tables else pd.DataFrame()

    return runs_df, raw_summary, raw_samples


runs_df, raw_summary, raw_samples = load_raw_outputs(RAW_RESULTS_DIR)

print("runs_df:", runs_df.shape)
print("raw_summary:", raw_summary.shape)
print("raw_samples:", raw_samples.shape)


runs_df: (17, 3)
raw_summary: (38, 34)
raw_samples: (31700, 34)


 Quick inspection

In [3]:
display(runs_df.head())
display(raw_summary.head())
display(raw_samples.head())


,model_alias_dir,run_id,run_dir
0,gemma3_4b_w4a16,2026-03-31_15-55-49,/home/stephane/resource-efficiency-benchmark/r...
1,gemma3_4b_w4a16,2026-04-02_03-43-28,/home/stephane/resource-efficiency-benchmark/r...
2,granite4_htiny_awq,2026-03-31_15-55-49,/home/stephane/resource-efficiency-benchmark/r...
3,granite4_htiny_awq,2026-04-02_03-43-28,/home/stephane/resource-efficiency-benchmark/r...
4,llama3_2_1b,2026-03-31_15-55-49,/home/stephane/resource-efficiency-benchmark/r...


,Model alias,Repo id,Served model id,Accuracy mean,Accuracy sd,TTFT ms mean,TTFT ms sd,End-to-end latency ms mean,End-to-end latency ms sd,Tokens/sec mean,Tokens/sec sd,CPU percent avg mean,CPU percent avg sd,Temp C avg mean,Temp C avg sd,GPU percent avg mean,GPU percent avg sd,Power W avg mean,Power W avg sd,Sequential sustained tokens/sec mean,Sequential sustained tokens/sec sd,Sequential sustained samples/sec mean,Sequential sustained samples/sec sd,CPU percent peak,RAM used MB peak,Process RAM MB peak,Temp C peak,GPU percent peak,VRAM used MB peak,Power W peak,model_alias_dir,run_id,dataset,source_path
0,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,0.52,0.0,71.9428,0.9192,127.9508,1.5443,18.2306,0.2049,7.4985,1.5035,70.0537,1.8942,12.8890,11.3311,60.7467,0.7789,5.5996,0.0661,5.4898,0.0648,65.0,14813.18,1019.99,79.500,29.0,6869.0,64.53,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...
1,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,0.26,0.0,70.8052,1.7534,124.9218,3.0983,18.5055,0.4667,7.0284,0.6785,69.5761,0.6263,11.2297,14.2129,61.1533,0.1697,5.6406,0.1397,5.6406,0.1397,61.0,12489.36,1020.02,71.625,34.0,6802.0,63.91,gemma3_4b_w4a16,2026-03-31_15-55-49,hellaswag,/home/stephane/resource-efficiency-benchmark/r...
2,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,0.03,0.0,71.8905,0.5932,151.7749,0.9190,44.4205,0.1135,7.4114,0.0777,72.2063,0.2803,32.8930,11.2241,101.1637,1.2629,22.2931,0.2932,4.5871,0.0603,29.3,8649.20,1020.02,74.250,100.0,6107.0,130.21,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,/home/stephane/resource-efficiency-benchmark/r...
3,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,NaN,NaN,89.8702,16.4796,623.5248,48.2390,87.6739,6.5252,8.9205,2.7985,72.6397,1.0871,55.9527,35.2792,166.9069,5.0177,70.3371,6.0829,1.4846,0.1025,44.1,10372.47,1192.01,75.375,100.0,4780.0,192.33,gemma3_4b_w4a16,2026-04-02_03-43-28,dolly,/home/stephane/resource-efficiency-benchmark/r...
4,granite4_htiny_awq,cyankiwi/granite-4.0-h-tiny-AWQ-4bit,Qwen/Qwen2.5-1.5B-Instruct,0.52,0.0,68.4738,0.2988,120.6718,0.4516,19.5530,0.0631,6.4220,0.0346,69.8841,0.4322,16.7760,13.7400,62.6146,1.0045,5.9502,0.0212,5.8335,0.0208,27.3,8697.27,1021.79,71.500,30.0,6197.0,66.89,granite4_htiny_awq,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...


,repo_id,served_model_id,model_alias,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,end_to_end_latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,gpu_util_percent_avg,gpu_util_percent_peak,vram_used_mb_peak,power_w_avg,power_w_peak,model_alias_dir,run_id,dataset,source_path
0,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,1,"In primates, the location of which of the foll...","In primates, the location of which of the foll...",the feet || the skull || the foramen magnum ||...,NaN,NaN,2.0,C,True,C. the foramen,3,128.2590,191.7984,47.2148,11.2000,14.1,14080.68,1019.95,74.2083,74.250,0.0,0.0,6709.0,51.7133,52.39,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...
1,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,2,In the comic strip 'Peanuts' what is Schroeder...,In the comic strip 'Peanuts' what is Schroeder...,Dancing || playing football || playing the pia...,NaN,NaN,2.0,C,True,C,1,79.7790,142.9208,15.8374,11.5667,12.6,14111.52,1019.95,74.3750,74.375,0.0,0.0,6709.0,54.2967,55.04,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...
2,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,3,The Yang-shao culture gave way to the Lung-Sha...,The Yang-shao culture gave way to the Lung-Sha...,"6,000 B.P. || 5,000 B.P. || 4,000 B.P. || 3,00...",NaN,NaN,1.0,C,False,C,1,81.9868,147.3174,15.3068,28.0000,57.5,14117.65,1019.95,74.3750,74.375,0.0,0.0,6709.0,56.6300,57.48,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...
3,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,4,At which of the following locations does bile ...,At which of the following locations does bile ...,Gastroesophageal sphincter || Duodenum || Ileo...,NaN,NaN,1.0,B,True,B,1,79.4288,143.5314,15.6000,27.2667,56.4,14123.26,1019.95,74.5417,74.625,0.0,0.0,6709.0,58.5467,59.84,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...
4,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,5,"Let A be a finite set with m elements, and let...","Let A be a finite set with m elements, and let...",n^m || n!/(n - m)! || n! || n!/(m!(n - m)!),NaN,NaN,0.0,A,True,A,1,82.5121,146.4888,15.6307,11.5333,13.2,14127.28,1019.95,74.6250,74.625,0.0,0.0,6709.0,60.6300,62.17,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...


 Normalize vLLM summary files

In [4]:
SUMMARY_MAP = {
    "Model alias": "model",
    "Repo id": "repo_id",
    "Served model id": "served_model_id",
    "Accuracy mean": "accuracy",
    "Accuracy sd": "accuracy_sd",
    "TTFT ms mean": "ttft",
    "TTFT ms sd": "ttft_sd",
    "End-to-end latency ms mean": "latency",
    "End-to-end latency ms sd": "latency_sd",
    "Tokens/sec mean": "tps",
    "Tokens/sec sd": "tps_sd",
    "CPU percent avg mean": "cpu",
    "CPU percent avg sd": "cpu_sd",
    "CPU percent peak": "cpu_peak",
    "RAM used MB peak": "ram_used_mb_peak",
    "Process RAM MB peak": "ram",
    "Temp C avg mean": "temp_c_avg",
    "Temp C avg sd": "temp_c_avg_sd",
    "Temp C peak": "temp_c_peak",
    "GPU percent avg mean": "gpu",
    "GPU percent avg sd": "gpu_sd",
    "GPU percent peak": "gpu_peak",
    "VRAM used MB peak": "vram",
    "Power W avg mean": "power",
    "Power W avg sd": "power_sd",
    "Power W peak": "power_peak",
    "Sequential sustained tokens/sec mean": "sustained_tps",
    "Sequential sustained tokens/sec sd": "sustained_tps_sd",
    "Sequential sustained samples/sec mean": "sustained_sps",
    "Sequential sustained samples/sec sd": "sustained_sps_sd",
}

def normalize_summary(df: pd.DataFrame) -> pd.DataFrame:
    out = []

    for _, r in df.iterrows():
        row = {
            "backend": "vllm",
            "run_id": r["run_id"],
            "dataset": r["dataset"],
            "model_alias_dir": r["model_alias_dir"],
            "source_path": r["source_path"],
        }

        working = dict(r)

        for raw_col, clean_col in SUMMARY_MAP.items():
            row[clean_col] = working.get(raw_col)

        if pd.isna(row.get("model")):
            row["model"] = r.get("model_alias_dir")

        out.append(row)

    summary = pd.DataFrame(out)

    numeric_cols = [
        "accuracy", "accuracy_sd",
        "ttft", "ttft_sd",
        "latency", "latency_sd",
        "tps", "tps_sd",
        "cpu", "cpu_sd", "cpu_peak",
        "ram_used_mb_peak", "ram",
        "temp_c_avg", "temp_c_avg_sd", "temp_c_peak",
        "gpu", "gpu_sd", "gpu_peak",
        "vram",
        "power", "power_sd", "power_peak",
        "sustained_tps", "sustained_tps_sd",
        "sustained_sps", "sustained_sps_sd",
    ]

    for col in numeric_cols:
        if col in summary.columns:
            summary[col] = pd.to_numeric(summary[col], errors="coerce")

    return summary


summary = normalize_summary(raw_summary)
summary.head()


,backend,run_id,dataset,model_alias_dir,source_path,model,repo_id,served_model_id,accuracy,accuracy_sd,ttft,ttft_sd,latency,latency_sd,tps,tps_sd,cpu,cpu_sd,cpu_peak,ram_used_mb_peak,ram,temp_c_avg,temp_c_avg_sd,temp_c_peak,gpu,gpu_sd,gpu_peak,vram,power,power_sd,power_peak,sustained_tps,sustained_tps_sd,sustained_sps,sustained_sps_sd
0,vllm,2026-03-31_15-55-49,mmlu,gemma3_4b_w4a16,/home/stephane/resource-efficiency-benchmark/r...,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,0.52,0.0,71.9428,0.9192,127.9508,1.5443,18.2306,0.2049,7.4985,1.5035,65.0,14813.18,1019.99,70.0537,1.8942,79.500,12.8890,11.3311,29.0,6869.0,60.7467,0.7789,64.53,5.5996,0.0661,5.4898,0.0648
1,vllm,2026-03-31_15-55-49,hellaswag,gemma3_4b_w4a16,/home/stephane/resource-efficiency-benchmark/r...,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,0.26,0.0,70.8052,1.7534,124.9218,3.0983,18.5055,0.4667,7.0284,0.6785,61.0,12489.36,1020.02,69.5761,0.6263,71.625,11.2297,14.2129,34.0,6802.0,61.1533,0.1697,63.91,5.6406,0.1397,5.6406,0.1397
2,vllm,2026-03-31_15-55-49,gsm8k,gemma3_4b_w4a16,/home/stephane/resource-efficiency-benchmark/r...,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,0.03,0.0,71.8905,0.5932,151.7749,0.9190,44.4205,0.1135,7.4114,0.0777,29.3,8649.20,1020.02,72.2063,0.2803,74.250,32.8930,11.2241,100.0,6107.0,101.1637,1.2629,130.21,22.2931,0.2932,4.5871,0.0603
3,vllm,2026-04-02_03-43-28,dolly,gemma3_4b_w4a16,/home/stephane/resource-efficiency-benchmark/r...,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,NaN,NaN,89.8702,16.4796,623.5248,48.2390,87.6739,6.5252,8.9205,2.7985,44.1,10372.47,1192.01,72.6397,1.0871,75.375,55.9527,35.2792,100.0,4780.0,166.9069,5.0177,192.33,70.3371,6.0829,1.4846,0.1025
4,vllm,2026-03-31_15-55-49,mmlu,granite4_htiny_awq,/home/stephane/resource-efficiency-benchmark/r...,granite4_htiny_awq,cyankiwi/granite-4.0-h-tiny-AWQ-4bit,Qwen/Qwen2.5-1.5B-Instruct,0.52,0.0,68.4738,0.2988,120.6718,0.4516,19.5530,0.0631,6.4220,0.0346,27.3,8697.27,1021.79,69.8841,0.4322,71.500,16.7760,13.7400,30.0,6197.0,62.6146,1.0045,66.89,5.9502,0.0212,5.8335,0.0208


 Normalize vLLM sample files

In [5]:
SAMPLE_RENAME_MAP = {
    "repo_id": "repo_id",
    "served_model_id": "served_model_id",
    "model_alias": "model",
    "dataset_name": "dataset_name",
    "run_idx": "run_idx",
    "sample_idx": "sample_idx",
    "prompt": "prompt",
    "question": "question",
    "choices": "choices",
    "instruction": "instruction",
    "context": "context",
    "gold_answer": "gold_answer",
    "parsed_prediction": "parsed_prediction",
    "correct": "correct",
    "raw_output": "raw_output",
    "output_tokens": "output_tokens",
    "ttft_ms": "ttft_ms",
    "end_to_end_latency_ms": "latency_ms",
    "tokens_per_sec": "tokens_per_sec",
    "cpu_percent_avg": "cpu_percent_avg",
    "cpu_percent_peak": "cpu_percent_peak",
    "ram_used_mb_peak": "ram_used_mb_peak",
    "process_ram_mb_peak": "process_ram_mb_peak",
    "temp_c_avg": "temp_c_avg",
    "temp_c_peak": "temp_c_peak",
    "gpu_util_percent_avg": "gpu_percent_avg",
    "gpu_util_percent_peak": "gpu_percent_peak",
    "vram_used_mb_peak": "vram_used_mb_peak",
    "power_w_avg": "power_w_avg",
    "power_w_peak": "power_w_peak",
}

def normalize_samples(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    keep_cols = [c for c in SAMPLE_RENAME_MAP if c in df.columns]
    base_cols = ["model_alias_dir", "run_id", "dataset", "source_path"]

    samples = df[keep_cols + base_cols].rename(columns=SAMPLE_RENAME_MAP).copy()
    samples["backend"] = "vllm"

    if "model" not in samples.columns:
        samples["model"] = samples["model_alias_dir"]

    numeric_cols = [
        "run_idx", "sample_idx", "output_tokens",
        "ttft_ms", "latency_ms", "tokens_per_sec",
        "cpu_percent_avg", "cpu_percent_peak",
        "ram_used_mb_peak", "process_ram_mb_peak",
        "temp_c_avg", "temp_c_peak",
        "gpu_percent_avg", "gpu_percent_peak",
        "vram_used_mb_peak",
        "power_w_avg", "power_w_peak",
    ]

    for col in numeric_cols:
        if col in samples.columns:
            samples[col] = pd.to_numeric(samples[col], errors="coerce")

    if "correct" in samples.columns:
        samples["correct"] = samples["correct"].map(
            lambda x: np.nan if pd.isna(x) else bool(x)
        )

    return samples


samples = normalize_samples(raw_samples)
samples.head()


,repo_id,served_model_id,model,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,gpu_percent_avg,gpu_percent_peak,vram_used_mb_peak,power_w_avg,power_w_peak,model_alias_dir,run_id,dataset,source_path,backend
0,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,1,"In primates, the location of which of the foll...","In primates, the location of which of the foll...",the feet || the skull || the foramen magnum ||...,NaN,NaN,2.0,C,True,C. the foramen,3,128.2590,191.7984,47.2148,11.2000,14.1,14080.68,1019.95,74.2083,74.250,0.0,0.0,6709.0,51.7133,52.39,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm
1,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,2,In the comic strip 'Peanuts' what is Schroeder...,In the comic strip 'Peanuts' what is Schroeder...,Dancing || playing football || playing the pia...,NaN,NaN,2.0,C,True,C,1,79.7790,142.9208,15.8374,11.5667,12.6,14111.52,1019.95,74.3750,74.375,0.0,0.0,6709.0,54.2967,55.04,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm
2,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,3,The Yang-shao culture gave way to the Lung-Sha...,The Yang-shao culture gave way to the Lung-Sha...,"6,000 B.P. || 5,000 B.P. || 4,000 B.P. || 3,00...",NaN,NaN,1.0,C,False,C,1,81.9868,147.3174,15.3068,28.0000,57.5,14117.65,1019.95,74.3750,74.375,0.0,0.0,6709.0,56.6300,57.48,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm
3,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,4,At which of the following locations does bile ...,At which of the following locations does bile ...,Gastroesophageal sphincter || Duodenum || Ileo...,NaN,NaN,1.0,B,True,B,1,79.4288,143.5314,15.6000,27.2667,56.4,14123.26,1019.95,74.5417,74.625,0.0,0.0,6709.0,58.5467,59.84,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm
4,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,5,"Let A be a finite set with m elements, and let...","Let A be a finite set with m elements, and let...",n^m || n!/(n - m)! || n! || n!/(m!(n - m)!),NaN,NaN,0.0,A,True,A,1,82.5121,146.4888,15.6307,11.5333,13.2,14127.28,1019.95,74.6250,74.625,0.0,0.0,6709.0,60.6300,62.17,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm


 Derived metrics on summary tables

In [6]:
def add_summary_metrics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["latency_s"] = out["latency"] / 1000.0
    out["ttft_s"] = out["ttft"] / 1000.0
    out["ttft_share_of_latency"] = out["ttft"] / out["latency"]

    out["tps_per_ram"] = out["tps"] / out["ram"]
    out["tps_per_vram"] = out["tps"] / out["vram"]
    out["tps_per_watt"] = out["tps"] / out["power"]
    out["sustained_tps_per_vram"] = out["sustained_tps"] / out["vram"]

    out["accuracy_per_latency"] = out["accuracy"] / out["latency"]
    out["accuracy_per_ram"] = out["accuracy"] / out["ram"]
    out["accuracy_per_vram"] = out["accuracy"] / out["vram"]
    out["accuracy_per_watt"] = out["accuracy"] / out["power"]

    return out


summary = add_summary_metrics(summary)
summary.head()


,backend,run_id,dataset,model_alias_dir,source_path,model,repo_id,served_model_id,accuracy,accuracy_sd,ttft,ttft_sd,latency,latency_sd,tps,tps_sd,cpu,cpu_sd,cpu_peak,ram_used_mb_peak,ram,temp_c_avg,temp_c_avg_sd,temp_c_peak,gpu,gpu_sd,gpu_peak,vram,power,power_sd,power_peak,sustained_tps,sustained_tps_sd,sustained_sps,sustained_sps_sd,latency_s,ttft_s,ttft_share_of_latency,tps_per_ram,tps_per_vram,tps_per_watt,sustained_tps_per_vram,accuracy_per_latency,accuracy_per_ram,accuracy_per_vram,accuracy_per_watt
0,vllm,2026-03-31_15-55-49,mmlu,gemma3_4b_w4a16,/home/stephane/resource-efficiency-benchmark/r...,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,0.52,0.0,71.9428,0.9192,127.9508,1.5443,18.2306,0.2049,7.4985,1.5035,65.0,14813.18,1019.99,70.0537,1.8942,79.500,12.8890,11.3311,29.0,6869.0,60.7467,0.7789,64.53,5.5996,0.0661,5.4898,0.0648,0.127951,0.071943,0.562269,0.017873,0.002654,0.300108,0.000815,0.004064,0.000510,0.000076,0.008560
1,vllm,2026-03-31_15-55-49,hellaswag,gemma3_4b_w4a16,/home/stephane/resource-efficiency-benchmark/r...,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,0.26,0.0,70.8052,1.7534,124.9218,3.0983,18.5055,0.4667,7.0284,0.6785,61.0,12489.36,1020.02,69.5761,0.6263,71.625,11.2297,14.2129,34.0,6802.0,61.1533,0.1697,63.91,5.6406,0.1397,5.6406,0.1397,0.124922,0.070805,0.566796,0.018142,0.002721,0.302608,0.000829,0.002081,0.000255,0.000038,0.004252
2,vllm,2026-03-31_15-55-49,gsm8k,gemma3_4b_w4a16,/home/stephane/resource-efficiency-benchmark/r...,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,0.03,0.0,71.8905,0.5932,151.7749,0.9190,44.4205,0.1135,7.4114,0.0777,29.3,8649.20,1020.02,72.2063,0.2803,74.250,32.8930,11.2241,100.0,6107.0,101.1637,1.2629,130.21,22.2931,0.2932,4.5871,0.0603,0.151775,0.071890,0.473665,0.043549,0.007274,0.439095,0.003650,0.000198,0.000029,0.000005,0.000297
3,vllm,2026-04-02_03-43-28,dolly,gemma3_4b_w4a16,/home/stephane/resource-efficiency-benchmark/r...,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,NaN,NaN,89.8702,16.4796,623.5248,48.2390,87.6739,6.5252,8.9205,2.7985,44.1,10372.47,1192.01,72.6397,1.0871,75.375,55.9527,35.2792,100.0,4780.0,166.9069,5.0177,192.33,70.3371,6.0829,1.4846,0.1025,0.623525,0.089870,0.144133,0.073551,0.018342,0.525286,0.014715,NaN,NaN,NaN,NaN
4,vllm,2026-03-31_15-55-49,mmlu,granite4_htiny_awq,/home/stephane/resource-efficiency-benchmark/r...,granite4_htiny_awq,cyankiwi/granite-4.0-h-tiny-AWQ-4bit,Qwen/Qwen2.5-1.5B-Instruct,0.52,0.0,68.4738,0.2988,120.6718,0.4516,19.5530,0.0631,6.4220,0.0346,27.3,8697.27,1021.79,69.8841,0.4322,71.500,16.7760,13.7400,30.0,6197.0,62.6146,1.0045,66.89,5.9502,0.0212,5.8335,0.0208,0.120672,0.068474,0.567438,0.019136,0.003155,0.312275,0.000960,0.004309,0.000509,0.000084,0.008305


 Derived metrics on sample tables

In [7]:
def add_sample_metrics(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["latency_s"] = out["latency_ms"] / 1000.0
    out["ttft_s"] = out["ttft_ms"] / 1000.0
    out["ttft_share_of_latency"] = out["ttft_ms"] / out["latency_ms"]

    out["tokens_per_ram"] = out["tokens_per_sec"] / out["process_ram_mb_peak"]
    out["tokens_per_vram"] = out["tokens_per_sec"] / out["vram_used_mb_peak"]
    out["tokens_per_watt"] = out["tokens_per_sec"] / out["power_w_avg"]

    return out


samples = add_sample_metrics(samples)
samples.head()


,repo_id,served_model_id,model,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,gpu_percent_avg,gpu_percent_peak,vram_used_mb_peak,power_w_avg,power_w_peak,model_alias_dir,run_id,dataset,source_path,backend,latency_s,ttft_s,ttft_share_of_latency,tokens_per_ram,tokens_per_vram,tokens_per_watt
0,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,1,"In primates, the location of which of the foll...","In primates, the location of which of the foll...",the feet || the skull || the foramen magnum ||...,NaN,NaN,2.0,C,True,C. the foramen,3,128.2590,191.7984,47.2148,11.2000,14.1,14080.68,1019.95,74.2083,74.250,0.0,0.0,6709.0,51.7133,52.39,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.191798,0.128259,0.668718,0.046291,0.007038,0.913011
1,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,2,In the comic strip 'Peanuts' what is Schroeder...,In the comic strip 'Peanuts' what is Schroeder...,Dancing || playing football || playing the pia...,NaN,NaN,2.0,C,True,C,1,79.7790,142.9208,15.8374,11.5667,12.6,14111.52,1019.95,74.3750,74.375,0.0,0.0,6709.0,54.2967,55.04,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.142921,0.079779,0.558204,0.015528,0.002361,0.291683
2,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,3,The Yang-shao culture gave way to the Lung-Sha...,The Yang-shao culture gave way to the Lung-Sha...,"6,000 B.P. || 5,000 B.P. || 4,000 B.P. || 3,00...",NaN,NaN,1.0,C,False,C,1,81.9868,147.3174,15.3068,28.0000,57.5,14117.65,1019.95,74.3750,74.375,0.0,0.0,6709.0,56.6300,57.48,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.147317,0.081987,0.556532,0.015007,0.002282,0.270295
3,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,4,At which of the following locations does bile ...,At which of the following locations does bile ...,Gastroesophageal sphincter || Duodenum || Ileo...,NaN,NaN,1.0,B,True,B,1,79.4288,143.5314,15.6000,27.2667,56.4,14123.26,1019.95,74.5417,74.625,0.0,0.0,6709.0,58.5467,59.84,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.143531,0.079429,0.553390,0.015295,0.002325,0.266454
4,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,5,"Let A be a finite set with m elements, and let...","Let A be a finite set with m elements, and let...",n^m || n!/(n - m)! || n! || n!/(m!(n - m)!),NaN,NaN,0.0,A,True,A,1,82.5121,146.4888,15.6307,11.5333,13.2,14127.28,1019.95,74.6250,74.625,0.0,0.0,6709.0,60.6300,62.17,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.146489,0.082512,0.563266,0.015325,0.002330,0.257805


 Split scored and unscored datasets

In [8]:
scored_datasets = ["mmlu", "hellaswag", "gsm8k"]
summary_scored = summary[summary["dataset"].isin(scored_datasets)].copy()
summary_dolly = summary[summary["dataset"] == "dolly"].copy()

samples_scored = samples[samples["dataset"].isin(scored_datasets)].copy()
samples_dolly = samples[samples["dataset"] == "dolly"].copy()

print("summary_scored:", summary_scored.shape)
print("summary_dolly:", summary_dolly.shape)
print("samples_scored:", samples_scored.shape)
print("samples_dolly:", samples_dolly.shape)


summary_scored: (27, 46)
summary_dolly: (11, 46)
samples_scored: (27000, 41)
samples_dolly: (4700, 41)


 Model x dataset summary

In [9]:
model_dataset_summary = (
    summary.groupby(["backend", "model", "dataset"], dropna=False)
    .agg(
        n_runs=("run_id", "nunique"),
        accuracy=("accuracy", "mean"),
        ttft=("ttft", "mean"),
        latency=("latency", "mean"),
        tps=("tps", "mean"),
        sustained_tps=("sustained_tps", "mean"),
        cpu=("cpu", "mean"),
        ram=("ram", "mean"),
        gpu=("gpu", "mean"),
        vram=("vram", "mean"),
        power=("power", "mean"),
        tps_per_vram=("tps_per_vram", "mean"),
        tps_per_watt=("tps_per_watt", "mean"),
    )
    .reset_index()
    .sort_values(["dataset", "model"])
)

display(model_dataset_summary)


,backend,model,dataset,n_runs,accuracy,ttft,latency,tps,sustained_tps,cpu,ram,gpu,vram,power,tps_per_vram,tps_per_watt
0,vllm,gemma3_4b_w4a16,dolly,1,NaN,89.87020,623.52480,87.67390,70.33710,8.92050,1192.01,55.9527,4780.0,166.90690,0.018342,0.525286
4,vllm,granite4_htiny_awq,dolly,1,NaN,89.29020,622.73140,87.74990,70.47460,8.16150,1192.37,60.5007,4804.0,170.06460,0.018266,0.515980
8,vllm,llama3_2_1b,dolly,1,NaN,89.70810,622.00140,87.88980,70.51030,8.25130,1192.16,40.3768,4812.0,169.06180,0.018265,0.519868
12,vllm,ministral3_3b_awq,dolly,1,NaN,89.47260,622.90910,87.78430,70.45230,8.35220,1192.23,55.2302,4812.0,169.71200,0.018243,0.517255
16,vllm,ministral3_8b_awq,dolly,2,NaN,84.16525,703.94195,83.05510,69.27895,10.14280,1107.23,62.0969,5643.0,170.83865,0.015175,0.486308
20,vllm,phi4mini_gptq_4bit,dolly,1,NaN,94.80220,635.63910,86.64310,67.51030,7.76470,1192.49,66.8110,4881.0,166.39580,0.017751,0.520705
24,vllm,qwen2_5_1_5b,dolly,2,NaN,85.36865,706.91390,82.85725,68.87145,9.39675,1106.86,64.2903,5502.0,169.70795,0.015413,0.488231
28,vllm,qwen2_7b_q4,dolly,1,NaN,81.23630,791.10150,77.92710,67.27420,10.62290,1021.78,81.0185,6214.0,169.95600,0.012541,0.458513
32,vllm,qwen3_4b_awq,dolly,1,NaN,81.34250,791.32440,77.93050,67.28970,10.61060,1021.65,85.0528,6194.0,169.83740,0.012582,0.458854
1,vllm,gemma3_4b_w4a16,gsm8k,1,0.03,71.89050,151.77490,44.42050,22.29310,7.41140,1020.02,32.8930,6107.0,101.16370,0.007274,0.439095


 Scored-task overview per model

In [10]:
scored_model_overview = (
    summary_scored.groupby(["backend", "model"], dropna=False)
    .agg(
        n_runs=("run_id", "nunique"),
        n_datasets=("dataset", "nunique"),
        avg_accuracy=("accuracy", "mean"),
        avg_ttft_ms=("ttft", "mean"),
        avg_latency_ms=("latency", "mean"),
        avg_tps=("tps", "mean"),
        avg_sustained_tps=("sustained_tps", "mean"),
        avg_cpu=("cpu", "mean"),
        avg_ram_mb=("ram", "mean"),
        avg_gpu=("gpu", "mean"),
        avg_vram_mb=("vram", "mean"),
        avg_power_w=("power", "mean"),
        avg_tps_per_vram=("tps_per_vram", "mean"),
        avg_tps_per_watt=("tps_per_watt", "mean"),
        avg_accuracy_per_latency=("accuracy_per_latency", "mean"),
        avg_accuracy_per_vram=("accuracy_per_vram", "mean"),
        avg_accuracy_per_watt=("accuracy_per_watt", "mean"),
    )
    .reset_index()
)

display(scored_model_overview.sort_values("avg_accuracy", ascending=False))


,backend,model,n_runs,n_datasets,avg_accuracy,avg_ttft_ms,avg_latency_ms,avg_tps,avg_sustained_tps,avg_cpu,avg_ram_mb,avg_gpu,avg_vram_mb,avg_power_w,avg_tps_per_vram,avg_tps_per_watt,avg_accuracy_per_latency,avg_accuracy_per_vram,avg_accuracy_per_watt
0,vllm,gemma3_4b_w4a16,1,3,0.27,71.546167,134.882500,27.052200,11.177767,7.312767,1020.010000,19.003900,6592.666667,74.354567,0.004216,0.347271,0.002114,0.000040,0.004369
1,vllm,granite4_htiny_awq,1,3,0.27,69.667400,131.040900,27.734200,11.344933,6.732467,1021.813333,20.150333,6195.666667,75.213667,0.004477,0.354035,0.002219,0.000044,0.004274
2,vllm,llama3_2_1b,1,3,0.27,69.863700,131.413967,27.670567,11.306467,6.739733,1020.890000,18.094000,6183.000000,75.192667,0.004476,0.353272,0.002215,0.000044,0.004267
3,vllm,ministral3_3b_awq,1,3,0.27,69.741600,131.283133,27.674500,11.320867,6.754767,1021.083333,20.730000,6209.333333,75.188667,0.004458,0.353252,0.002216,0.000044,0.004274
4,vllm,ministral3_8b_awq,1,3,0.27,69.826467,131.393633,27.648500,11.310733,6.758667,1021.966667,16.514900,6210.000000,75.030700,0.004445,0.353840,0.002211,0.000044,0.004287
5,vllm,phi4mini_gptq_4bit,1,3,0.27,72.752167,137.666833,26.416900,10.943933,7.186167,1016.476667,15.277333,6664.333333,66.823900,0.003957,0.382358,0.002094,0.000041,0.004294
6,vllm,qwen2_5_1_5b,1,3,0.27,69.573967,130.966800,27.727800,11.369767,6.735733,1021.236667,17.233767,6196.000000,75.293800,0.004476,0.353518,0.002219,0.000044,0.004277
7,vllm,qwen2_7b_q4,1,3,0.27,69.679700,131.080133,27.722433,11.343367,6.745267,1021.683333,15.415233,6194.000000,75.208600,0.004476,0.353915,0.002219,0.000044,0.004271
8,vllm,qwen3_4b_awq,1,3,0.27,69.793967,131.292733,27.686533,11.322567,6.750267,1021.440000,15.571667,6195.000000,75.191567,0.004470,0.353427,0.002214,0.000044,0.004274


 Dolly / generation overview per model

In [11]:
dolly_model_overview = (
    summary_dolly.groupby(["backend", "model"], dropna=False)
    .agg(
        n_runs=("run_id", "nunique"),
        avg_ttft_ms=("ttft", "mean"),
        avg_latency_ms=("latency", "mean"),
        avg_tps=("tps", "mean"),
        avg_sustained_tps=("sustained_tps", "mean"),
        avg_cpu=("cpu", "mean"),
        avg_ram_mb=("ram", "mean"),
        avg_gpu=("gpu", "mean"),
        avg_vram_mb=("vram", "mean"),
        avg_power_w=("power", "mean"),
        avg_tps_per_vram=("tps_per_vram", "mean"),
        avg_tps_per_watt=("tps_per_watt", "mean"),
    )
    .reset_index()
)

display(dolly_model_overview.sort_values("avg_tps", ascending=False))


,backend,model,n_runs,avg_ttft_ms,avg_latency_ms,avg_tps,avg_sustained_tps,avg_cpu,avg_ram_mb,avg_gpu,avg_vram_mb,avg_power_w,avg_tps_per_vram,avg_tps_per_watt
2,vllm,llama3_2_1b,1,89.70810,622.00140,87.88980,70.51030,8.25130,1192.16,40.3768,4812.0,169.06180,0.018265,0.519868
3,vllm,ministral3_3b_awq,1,89.47260,622.90910,87.78430,70.45230,8.35220,1192.23,55.2302,4812.0,169.71200,0.018243,0.517255
1,vllm,granite4_htiny_awq,1,89.29020,622.73140,87.74990,70.47460,8.16150,1192.37,60.5007,4804.0,170.06460,0.018266,0.515980
0,vllm,gemma3_4b_w4a16,1,89.87020,623.52480,87.67390,70.33710,8.92050,1192.01,55.9527,4780.0,166.90690,0.018342,0.525286
5,vllm,phi4mini_gptq_4bit,1,94.80220,635.63910,86.64310,67.51030,7.76470,1192.49,66.8110,4881.0,166.39580,0.017751,0.520705
4,vllm,ministral3_8b_awq,2,84.16525,703.94195,83.05510,69.27895,10.14280,1107.23,62.0969,5643.0,170.83865,0.015175,0.486308
6,vllm,qwen2_5_1_5b,2,85.36865,706.91390,82.85725,68.87145,9.39675,1106.86,64.2903,5502.0,169.70795,0.015413,0.488231
8,vllm,qwen3_4b_awq,1,81.34250,791.32440,77.93050,67.28970,10.61060,1021.65,85.0528,6194.0,169.83740,0.012582,0.458854
7,vllm,qwen2_7b_q4,1,81.23630,791.10150,77.92710,67.27420,10.62290,1021.78,81.0185,6214.0,169.95600,0.012541,0.458513


 Run-level stability from samples

In [12]:
run_level_from_samples = (
    samples.groupby(["backend", "model", "run_id", "dataset", "run_idx"], dropna=False)
    .agg(
        n_samples=("sample_idx", "count"),
        mean_output_tokens=("output_tokens", "mean"),
        mean_ttft_ms=("ttft_ms", "mean"),
        mean_latency_ms=("latency_ms", "mean"),
        mean_tokens_per_sec=("tokens_per_sec", "mean"),
        mean_cpu=("cpu_percent_avg", "mean"),
        mean_ram_mb=("process_ram_mb_peak", "mean"),
        mean_gpu=("gpu_percent_avg", "mean"),
        mean_vram_mb=("vram_used_mb_peak", "mean"),
        mean_power_w=("power_w_avg", "mean"),
        accuracy=("correct", "mean"),
    )
    .reset_index()
)

display(run_level_from_samples.head(20))


,backend,model,run_id,dataset,run_idx,n_samples,mean_output_tokens,mean_ttft_ms,mean_latency_ms,mean_tokens_per_sec,mean_cpu,mean_ram_mb,mean_gpu,mean_vram_mb,mean_power_w,accuracy
0,vllm,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,1,100,4.86,71.698808,151.264240,44.513852,7.454668,1020.0200,13.763331,6107.00,98.631133,0.03
1,vllm,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,2,100,4.86,72.094976,152.023525,44.428220,7.351331,1020.0200,48.896667,6107.00,99.638828,0.03
2,vllm,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,3,100,4.86,71.299260,150.838795,44.562966,7.471671,1020.0200,45.546667,6098.42,100.786632,0.03
3,vllm,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,4,100,4.86,71.542321,151.156742,44.503135,7.454337,1020.0200,34.270000,6094.00,101.391065,0.03
4,vllm,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,5,100,4.86,72.256836,152.230166,44.390346,7.379668,1020.0200,31.969999,6094.00,101.574667,0.03
5,vllm,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,6,100,4.86,71.416370,151.014199,44.495626,7.423000,1020.0200,28.166666,6094.00,101.592969,0.03
6,vllm,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,7,100,4.86,73.160833,153.703200,44.201914,7.268335,1020.0200,17.396668,6094.00,100.854764,0.03
7,vllm,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,8,100,4.86,72.410685,152.763645,44.269333,7.328336,1020.0200,31.966666,6094.00,102.686699,0.03
8,vllm,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,9,100,4.86,71.302133,151.043632,44.457369,7.524006,1020.0200,41.883333,6093.26,101.829870,0.03
9,vllm,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,10,100,4.86,71.722815,151.710767,44.382311,7.458334,1020.0200,35.070001,6092.00,102.650335,0.03


 Incorrect scored samples

In [13]:
incorrect_samples = samples_scored[samples_scored["correct"] == False].copy()

cols_to_show = [
    "model", "run_id", "dataset", "run_idx", "sample_idx",
    "question", "gold_answer", "parsed_prediction",
    "raw_output", "ttft_ms", "latency_ms", "tokens_per_sec",
    "gpu_percent_avg", "vram_used_mb_peak", "power_w_avg",
]
cols_to_show = [c for c in cols_to_show if c in incorrect_samples.columns]

display(incorrect_samples[cols_to_show].head(50))


,model,run_id,dataset,run_idx,sample_idx,question,gold_answer,parsed_prediction,raw_output,ttft_ms,latency_ms,tokens_per_sec,gpu_percent_avg,vram_used_mb_peak,power_w_avg
2,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,1,3,The Yang-shao culture gave way to the Lung-Sha...,1.0,C,C,81.9868,147.3174,15.3068,0.0,6709.0,56.6300
6,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,1,7,A state imposes a graduated income tax upon ne...,2.0,A,A,72.9912,129.2704,17.7686,0.0,6709.0,60.8467
7,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,1,8,What is personal (ratione personae) immunity?,2.0,B,B,73.3081,131.8405,17.0846,0.0,6709.0,61.0267
8,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,1,9,What is the meaning of collective security?,2.0,A,A,73.1569,129.4050,17.7784,0.0,6709.0,62.0400
10,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,1,11,Let T be a depth-first search tree of a connec...,3.0,C,C,73.1867,131.7363,17.0795,0.0,6709.0,62.7733
11,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,1,12,Including the bottom how many sides are on a s...,2.0,B,B,73.0379,129.4949,17.7126,0.0,6709.0,62.7733
12,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,1,13,Anscombe criticizes as absurd Kant’s idea of:,3.0,A,A,72.3892,130.1284,17.3192,0.0,6709.0,62.7900
15,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,1,16,A disk of radius 0.1 m initially at rest under...,3.0,C,C,70.7907,125.5176,18.2725,0.0,6709.0,62.7800
20,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,1,21,For which of these two scenarios does the main...,2.0,A,A,74.9930,135.1213,16.6311,0.0,6709.0,62.3133
22,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,1,23,A member of the president's cabinet is said to...,2.0,A,A,74.0507,131.8006,17.3160,0.0,6709.0,62.7300


 Sample-level aggregate summary

In [14]:
sample_aggregate_summary = (
    samples.groupby(["backend", "model", "dataset"], dropna=False)
    .agg(
        n_rows=("sample_idx", "count"),
        mean_output_tokens=("output_tokens", "mean"),
        mean_ttft_ms=("ttft_ms", "mean"),
        mean_latency_ms=("latency_ms", "mean"),
        mean_tokens_per_sec=("tokens_per_sec", "mean"),
        mean_cpu=("cpu_percent_avg", "mean"),
        mean_ram_mb=("process_ram_mb_peak", "mean"),
        mean_gpu=("gpu_percent_avg", "mean"),
        mean_vram_mb=("vram_used_mb_peak", "mean"),
        mean_power_w=("power_w_avg", "mean"),
        mean_tokens_per_vram=("tokens_per_vram", "mean"),
        mean_tokens_per_watt=("tokens_per_watt", "mean"),
        pct_correct=("correct", "mean"),
    )
    .reset_index()
    .sort_values(["dataset", "model"])
)

display(sample_aggregate_summary)


,backend,model,dataset,n_rows,mean_output_tokens,mean_ttft_ms,mean_latency_ms,mean_tokens_per_sec,mean_cpu,mean_ram_mb,mean_gpu,mean_vram_mb,mean_power_w,mean_tokens_per_vram,mean_tokens_per_watt,pct_correct
0,vllm,gemma3_4b_w4a16,dolly,100,47.580000,89.870169,623.524775,87.673926,8.920451,1191.246300,55.952667,4655.580000,166.906925,0.018830,0.554761,NaN
4,vllm,granite4_htiny_awq,dolly,100,47.580000,89.290217,622.731401,87.749866,8.161501,1192.340500,60.500666,4648.800000,170.064604,0.018875,0.544111,NaN
8,vllm,llama3_2_1b,dolly,100,47.580000,89.708098,622.001359,87.889758,8.251334,1192.065800,40.376833,4656.300000,169.061784,0.018875,0.549991,NaN
12,vllm,ministral3_3b_awq,dolly,100,47.580000,89.472610,622.909119,87.784339,8.352236,1192.181300,55.230166,4654.590000,169.712035,0.018863,0.547474,NaN
16,vllm,ministral3_8b_awq,dolly,1100,56.517273,80.051009,770.134705,79.297562,11.628853,1037.499391,77.139134,6239.642727,171.538000,0.012865,0.467436,NaN
20,vllm,phi4mini_gptq_4bit,dolly,100,47.580000,94.802207,635.639091,86.643097,7.764682,1192.454600,66.810999,4692.330000,166.395786,0.018467,0.544722,NaN
24,vllm,qwen2_5_1_5b,dolly,1100,56.466364,81.894069,775.464909,78.874747,10.391384,1036.883064,76.204689,6052.232727,169.693584,0.013164,0.470013,NaN
28,vllm,qwen2_7b_q4,dolly,1000,57.355000,81.236344,791.101535,77.927114,10.622864,1021.758250,81.018451,6192.511000,169.956018,0.012584,0.461240,NaN
32,vllm,qwen3_4b_awq,dolly,1000,57.355000,81.342534,791.324360,77.930460,10.610634,1021.545540,85.052822,6192.745000,169.837351,0.012584,0.461659,NaN
1,vllm,gemma3_4b_w4a16,gsm8k,1000,4.860000,71.890504,151.774891,44.420507,7.411369,1020.020000,32.893000,6096.768000,101.163696,0.007286,0.441166,0.03


 Rankings

In [15]:
ranking_accuracy = scored_model_overview.sort_values(
    ["avg_accuracy", "avg_tps_per_watt", "avg_tps_per_vram"],
    ascending=[False, False, False]
).reset_index(drop=True)

ranking_efficiency = scored_model_overview.sort_values(
    ["avg_tps_per_watt", "avg_accuracy", "avg_tps_per_vram"],
    ascending=[False, False, False]
).reset_index(drop=True)

display(ranking_accuracy)
display(ranking_efficiency)


,backend,model,n_runs,n_datasets,avg_accuracy,avg_ttft_ms,avg_latency_ms,avg_tps,avg_sustained_tps,avg_cpu,avg_ram_mb,avg_gpu,avg_vram_mb,avg_power_w,avg_tps_per_vram,avg_tps_per_watt,avg_accuracy_per_latency,avg_accuracy_per_vram,avg_accuracy_per_watt
0,vllm,phi4mini_gptq_4bit,1,3,0.27,72.752167,137.666833,26.416900,10.943933,7.186167,1016.476667,15.277333,6664.333333,66.823900,0.003957,0.382358,0.002094,0.000041,0.004294
1,vllm,granite4_htiny_awq,1,3,0.27,69.667400,131.040900,27.734200,11.344933,6.732467,1021.813333,20.150333,6195.666667,75.213667,0.004477,0.354035,0.002219,0.000044,0.004274
2,vllm,qwen2_7b_q4,1,3,0.27,69.679700,131.080133,27.722433,11.343367,6.745267,1021.683333,15.415233,6194.000000,75.208600,0.004476,0.353915,0.002219,0.000044,0.004271
3,vllm,ministral3_8b_awq,1,3,0.27,69.826467,131.393633,27.648500,11.310733,6.758667,1021.966667,16.514900,6210.000000,75.030700,0.004445,0.353840,0.002211,0.000044,0.004287
4,vllm,qwen2_5_1_5b,1,3,0.27,69.573967,130.966800,27.727800,11.369767,6.735733,1021.236667,17.233767,6196.000000,75.293800,0.004476,0.353518,0.002219,0.000044,0.004277
5,vllm,qwen3_4b_awq,1,3,0.27,69.793967,131.292733,27.686533,11.322567,6.750267,1021.440000,15.571667,6195.000000,75.191567,0.004470,0.353427,0.002214,0.000044,0.004274
6,vllm,llama3_2_1b,1,3,0.27,69.863700,131.413967,27.670567,11.306467,6.739733,1020.890000,18.094000,6183.000000,75.192667,0.004476,0.353272,0.002215,0.000044,0.004267
7,vllm,ministral3_3b_awq,1,3,0.27,69.741600,131.283133,27.674500,11.320867,6.754767,1021.083333,20.730000,6209.333333,75.188667,0.004458,0.353252,0.002216,0.000044,0.004274
8,vllm,gemma3_4b_w4a16,1,3,0.27,71.546167,134.882500,27.052200,11.177767,7.312767,1020.010000,19.003900,6592.666667,74.354567,0.004216,0.347271,0.002114,0.000040,0.004369


,backend,model,n_runs,n_datasets,avg_accuracy,avg_ttft_ms,avg_latency_ms,avg_tps,avg_sustained_tps,avg_cpu,avg_ram_mb,avg_gpu,avg_vram_mb,avg_power_w,avg_tps_per_vram,avg_tps_per_watt,avg_accuracy_per_latency,avg_accuracy_per_vram,avg_accuracy_per_watt
0,vllm,phi4mini_gptq_4bit,1,3,0.27,72.752167,137.666833,26.416900,10.943933,7.186167,1016.476667,15.277333,6664.333333,66.823900,0.003957,0.382358,0.002094,0.000041,0.004294
1,vllm,granite4_htiny_awq,1,3,0.27,69.667400,131.040900,27.734200,11.344933,6.732467,1021.813333,20.150333,6195.666667,75.213667,0.004477,0.354035,0.002219,0.000044,0.004274
2,vllm,qwen2_7b_q4,1,3,0.27,69.679700,131.080133,27.722433,11.343367,6.745267,1021.683333,15.415233,6194.000000,75.208600,0.004476,0.353915,0.002219,0.000044,0.004271
3,vllm,ministral3_8b_awq,1,3,0.27,69.826467,131.393633,27.648500,11.310733,6.758667,1021.966667,16.514900,6210.000000,75.030700,0.004445,0.353840,0.002211,0.000044,0.004287
4,vllm,qwen2_5_1_5b,1,3,0.27,69.573967,130.966800,27.727800,11.369767,6.735733,1021.236667,17.233767,6196.000000,75.293800,0.004476,0.353518,0.002219,0.000044,0.004277
5,vllm,qwen3_4b_awq,1,3,0.27,69.793967,131.292733,27.686533,11.322567,6.750267,1021.440000,15.571667,6195.000000,75.191567,0.004470,0.353427,0.002214,0.000044,0.004274
6,vllm,llama3_2_1b,1,3,0.27,69.863700,131.413967,27.670567,11.306467,6.739733,1020.890000,18.094000,6183.000000,75.192667,0.004476,0.353272,0.002215,0.000044,0.004267
7,vllm,ministral3_3b_awq,1,3,0.27,69.741600,131.283133,27.674500,11.320867,6.754767,1021.083333,20.730000,6209.333333,75.188667,0.004458,0.353252,0.002216,0.000044,0.004274
8,vllm,gemma3_4b_w4a16,1,3,0.27,71.546167,134.882500,27.052200,11.177767,7.312767,1020.010000,19.003900,6592.666667,74.354567,0.004216,0.347271,0.002114,0.000040,0.004369


 Output paths

In [16]:
summary_normalized_path = PROCESSED_DIR / "summary_vllm.csv"
summary_scored_path = PROCESSED_DIR / "summary_scored_vllm.csv"
summary_dolly_path = PROCESSED_DIR / "summary_dolly_vllm.csv"

samples_all_path = PROCESSED_DIR / "samples_vllm.csv"
samples_scored_path = PROCESSED_DIR / "samples_scored_vllm.csv"
samples_dolly_path = PROCESSED_DIR / "samples_dolly_vllm.csv"

model_dataset_summary_path = PROCESSED_DIR / "model_dataset_summary_vllm.csv"
scored_model_overview_path = PROCESSED_DIR / "scored_model_overview_vllm.csv"
dolly_model_overview_path = PROCESSED_DIR / "dolly_model_overview_vllm.csv"
run_level_from_samples_path = PROCESSED_DIR / "run_level_from_samples_vllm.csv"
sample_aggregate_summary_path = PROCESSED_DIR / "sample_aggregate_summary_vllm.csv"
incorrect_samples_path = PROCESSED_DIR / "incorrect_samples_vllm.csv"
ranking_accuracy_path = PROCESSED_DIR / "ranking_accuracy_vllm.csv"
ranking_efficiency_path = PROCESSED_DIR / "ranking_efficiency_vllm.csv"


 Save processed outputs

In [17]:
summary.to_csv(summary_normalized_path, index=False)
summary_scored.to_csv(summary_scored_path, index=False)
summary_dolly.to_csv(summary_dolly_path, index=False)

samples.to_csv(samples_all_path, index=False)
samples_scored.to_csv(samples_scored_path, index=False)
samples_dolly.to_csv(samples_dolly_path, index=False)

model_dataset_summary.to_csv(model_dataset_summary_path, index=False)
scored_model_overview.to_csv(scored_model_overview_path, index=False)
dolly_model_overview.to_csv(dolly_model_overview_path, index=False)
run_level_from_samples.to_csv(run_level_from_samples_path, index=False)
sample_aggregate_summary.to_csv(sample_aggregate_summary_path, index=False)
incorrect_samples.to_csv(incorrect_samples_path, index=False)
ranking_accuracy.to_csv(ranking_accuracy_path, index=False)
ranking_efficiency.to_csv(ranking_efficiency_path, index=False)

print("Saved:")
for p in [
    summary_normalized_path,
    summary_scored_path,
    summary_dolly_path,
    samples_all_path,
    samples_scored_path,
    samples_dolly_path,
    model_dataset_summary_path,
    scored_model_overview_path,
    dolly_model_overview_path,
    run_level_from_samples_path,
    sample_aggregate_summary_path,
    incorrect_samples_path,
    ranking_accuracy_path,
    ranking_efficiency_path,
]:
    print("-", p.name)


Saved:
- summary_vllm.csv
- summary_scored_vllm.csv
- summary_dolly_vllm.csv
- samples_vllm.csv
- samples_scored_vllm.csv
- samples_dolly_vllm.csv
- model_dataset_summary_vllm.csv
- scored_model_overview_vllm.csv
- dolly_model_overview_vllm.csv
- run_level_from_samples_vllm.csv
- sample_aggregate_summary_vllm.csv
- incorrect_samples_vllm.csv
- ranking_accuracy_vllm.csv
- ranking_efficiency_vllm.csv


 Final inspection

In [18]:
display(summary.head())
display(samples.head())
display(ranking_accuracy.head())
display(ranking_efficiency.head())

,backend,run_id,dataset,model_alias_dir,source_path,model,repo_id,served_model_id,accuracy,accuracy_sd,ttft,ttft_sd,latency,latency_sd,tps,tps_sd,cpu,cpu_sd,cpu_peak,ram_used_mb_peak,ram,temp_c_avg,temp_c_avg_sd,temp_c_peak,gpu,gpu_sd,gpu_peak,vram,power,power_sd,power_peak,sustained_tps,sustained_tps_sd,sustained_sps,sustained_sps_sd,latency_s,ttft_s,ttft_share_of_latency,tps_per_ram,tps_per_vram,tps_per_watt,sustained_tps_per_vram,accuracy_per_latency,accuracy_per_ram,accuracy_per_vram,accuracy_per_watt
0,vllm,2026-03-31_15-55-49,mmlu,gemma3_4b_w4a16,/home/stephane/resource-efficiency-benchmark/r...,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,0.52,0.0,71.9428,0.9192,127.9508,1.5443,18.2306,0.2049,7.4985,1.5035,65.0,14813.18,1019.99,70.0537,1.8942,79.500,12.8890,11.3311,29.0,6869.0,60.7467,0.7789,64.53,5.5996,0.0661,5.4898,0.0648,0.127951,0.071943,0.562269,0.017873,0.002654,0.300108,0.000815,0.004064,0.000510,0.000076,0.008560
1,vllm,2026-03-31_15-55-49,hellaswag,gemma3_4b_w4a16,/home/stephane/resource-efficiency-benchmark/r...,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,0.26,0.0,70.8052,1.7534,124.9218,3.0983,18.5055,0.4667,7.0284,0.6785,61.0,12489.36,1020.02,69.5761,0.6263,71.625,11.2297,14.2129,34.0,6802.0,61.1533,0.1697,63.91,5.6406,0.1397,5.6406,0.1397,0.124922,0.070805,0.566796,0.018142,0.002721,0.302608,0.000829,0.002081,0.000255,0.000038,0.004252
2,vllm,2026-03-31_15-55-49,gsm8k,gemma3_4b_w4a16,/home/stephane/resource-efficiency-benchmark/r...,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,0.03,0.0,71.8905,0.5932,151.7749,0.9190,44.4205,0.1135,7.4114,0.0777,29.3,8649.20,1020.02,72.2063,0.2803,74.250,32.8930,11.2241,100.0,6107.0,101.1637,1.2629,130.21,22.2931,0.2932,4.5871,0.0603,0.151775,0.071890,0.473665,0.043549,0.007274,0.439095,0.003650,0.000198,0.000029,0.000005,0.000297
3,vllm,2026-04-02_03-43-28,dolly,gemma3_4b_w4a16,/home/stephane/resource-efficiency-benchmark/r...,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,NaN,NaN,89.8702,16.4796,623.5248,48.2390,87.6739,6.5252,8.9205,2.7985,44.1,10372.47,1192.01,72.6397,1.0871,75.375,55.9527,35.2792,100.0,4780.0,166.9069,5.0177,192.33,70.3371,6.0829,1.4846,0.1025,0.623525,0.089870,0.144133,0.073551,0.018342,0.525286,0.014715,NaN,NaN,NaN,NaN
4,vllm,2026-03-31_15-55-49,mmlu,granite4_htiny_awq,/home/stephane/resource-efficiency-benchmark/r...,granite4_htiny_awq,cyankiwi/granite-4.0-h-tiny-AWQ-4bit,Qwen/Qwen2.5-1.5B-Instruct,0.52,0.0,68.4738,0.2988,120.6718,0.4516,19.5530,0.0631,6.4220,0.0346,27.3,8697.27,1021.79,69.8841,0.4322,71.500,16.7760,13.7400,30.0,6197.0,62.6146,1.0045,66.89,5.9502,0.0212,5.8335,0.0208,0.120672,0.068474,0.567438,0.019136,0.003155,0.312275,0.000960,0.004309,0.000509,0.000084,0.008305


,repo_id,served_model_id,model,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,gpu_percent_avg,gpu_percent_peak,vram_used_mb_peak,power_w_avg,power_w_peak,model_alias_dir,run_id,dataset,source_path,backend,latency_s,ttft_s,ttft_share_of_latency,tokens_per_ram,tokens_per_vram,tokens_per_watt
0,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,1,"In primates, the location of which of the foll...","In primates, the location of which of the foll...",the feet || the skull || the foramen magnum ||...,NaN,NaN,2.0,C,True,C. the foramen,3,128.2590,191.7984,47.2148,11.2000,14.1,14080.68,1019.95,74.2083,74.250,0.0,0.0,6709.0,51.7133,52.39,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.191798,0.128259,0.668718,0.046291,0.007038,0.913011
1,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,2,In the comic strip 'Peanuts' what is Schroeder...,In the comic strip 'Peanuts' what is Schroeder...,Dancing || playing football || playing the pia...,NaN,NaN,2.0,C,True,C,1,79.7790,142.9208,15.8374,11.5667,12.6,14111.52,1019.95,74.3750,74.375,0.0,0.0,6709.0,54.2967,55.04,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.142921,0.079779,0.558204,0.015528,0.002361,0.291683
2,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,3,The Yang-shao culture gave way to the Lung-Sha...,The Yang-shao culture gave way to the Lung-Sha...,"6,000 B.P. || 5,000 B.P. || 4,000 B.P. || 3,00...",NaN,NaN,1.0,C,False,C,1,81.9868,147.3174,15.3068,28.0000,57.5,14117.65,1019.95,74.3750,74.375,0.0,0.0,6709.0,56.6300,57.48,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.147317,0.081987,0.556532,0.015007,0.002282,0.270295
3,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,4,At which of the following locations does bile ...,At which of the following locations does bile ...,Gastroesophageal sphincter || Duodenum || Ileo...,NaN,NaN,1.0,B,True,B,1,79.4288,143.5314,15.6000,27.2667,56.4,14123.26,1019.95,74.5417,74.625,0.0,0.0,6709.0,58.5467,59.84,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.143531,0.079429,0.553390,0.015295,0.002325,0.266454
4,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,5,"Let A be a finite set with m elements, and let...","Let A be a finite set with m elements, and let...",n^m || n!/(n - m)! || n! || n!/(m!(n - m)!),NaN,NaN,0.0,A,True,A,1,82.5121,146.4888,15.6307,11.5333,13.2,14127.28,1019.95,74.6250,74.625,0.0,0.0,6709.0,60.6300,62.17,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...,vllm,0.146489,0.082512,0.563266,0.015325,0.002330,0.257805


,backend,model,n_runs,n_datasets,avg_accuracy,avg_ttft_ms,avg_latency_ms,avg_tps,avg_sustained_tps,avg_cpu,avg_ram_mb,avg_gpu,avg_vram_mb,avg_power_w,avg_tps_per_vram,avg_tps_per_watt,avg_accuracy_per_latency,avg_accuracy_per_vram,avg_accuracy_per_watt
0,vllm,phi4mini_gptq_4bit,1,3,0.27,72.752167,137.666833,26.416900,10.943933,7.186167,1016.476667,15.277333,6664.333333,66.823900,0.003957,0.382358,0.002094,0.000041,0.004294
1,vllm,granite4_htiny_awq,1,3,0.27,69.667400,131.040900,27.734200,11.344933,6.732467,1021.813333,20.150333,6195.666667,75.213667,0.004477,0.354035,0.002219,0.000044,0.004274
2,vllm,qwen2_7b_q4,1,3,0.27,69.679700,131.080133,27.722433,11.343367,6.745267,1021.683333,15.415233,6194.000000,75.208600,0.004476,0.353915,0.002219,0.000044,0.004271
3,vllm,ministral3_8b_awq,1,3,0.27,69.826467,131.393633,27.648500,11.310733,6.758667,1021.966667,16.514900,6210.000000,75.030700,0.004445,0.353840,0.002211,0.000044,0.004287
4,vllm,qwen2_5_1_5b,1,3,0.27,69.573967,130.966800,27.727800,11.369767,6.735733,1021.236667,17.233767,6196.000000,75.293800,0.004476,0.353518,0.002219,0.000044,0.004277


,backend,model,n_runs,n_datasets,avg_accuracy,avg_ttft_ms,avg_latency_ms,avg_tps,avg_sustained_tps,avg_cpu,avg_ram_mb,avg_gpu,avg_vram_mb,avg_power_w,avg_tps_per_vram,avg_tps_per_watt,avg_accuracy_per_latency,avg_accuracy_per_vram,avg_accuracy_per_watt
0,vllm,phi4mini_gptq_4bit,1,3,0.27,72.752167,137.666833,26.416900,10.943933,7.186167,1016.476667,15.277333,6664.333333,66.823900,0.003957,0.382358,0.002094,0.000041,0.004294
1,vllm,granite4_htiny_awq,1,3,0.27,69.667400,131.040900,27.734200,11.344933,6.732467,1021.813333,20.150333,6195.666667,75.213667,0.004477,0.354035,0.002219,0.000044,0.004274
2,vllm,qwen2_7b_q4,1,3,0.27,69.679700,131.080133,27.722433,11.343367,6.745267,1021.683333,15.415233,6194.000000,75.208600,0.004476,0.353915,0.002219,0.000044,0.004271
3,vllm,ministral3_8b_awq,1,3,0.27,69.826467,131.393633,27.648500,11.310733,6.758667,1021.966667,16.514900,6210.000000,75.030700,0.004445,0.353840,0.002211,0.000044,0.004287
4,vllm,qwen2_5_1_5b,1,3,0.27,69.573967,130.966800,27.727800,11.369767,6.735733,1021.236667,17.233767,6196.000000,75.293800,0.004476,0.353518,0.002219,0.000044,0.004277
